In [1]:
import os
import sys
import torch
import cv2

# 将父目录加入 path 以便导入 drone_dynamics 和 drone_renderer
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from drone_dynamics import simulate_position_step, solve_attitude_from_thrust_and_goal_vec,update_dg
from drone_renderer import DroneRenderer

# 检查 CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


In [2]:
B = 4  # 批量大小
dt = 0.02  # 时间步长
num_steps = 200  # 模拟步数

mesh_path = "../data/sample/sample.obj"  # 无人机模型路径
renderer = DroneRenderer(
    mesh_path=mesh_path,
    device=device,
    image_size=(480, 640),
    focal_length=500.0
)

# 初始化无人机状态
p = torch.rand(B, 3, device=device) * 2.0  # 位置
print(p.shape)
v = torch.zeros(B, 3, device=device)  # 速度
a = torch.zeros(B, 3, device=device)  # 加速度
act = torch.zeros(B, 3, device=device)  # 动力分配 (实际物理推力状态)
R = torch.eye(3, device=device).unsqueeze(0).repeat(B, 1, 1)  # 姿态矩阵

# 随机生成一个恒定的期望指令用于测试
act_pred_target = torch.rand(B, 3, device=device) * 15.0 
dg = torch.randn((B, 3), device=device) * 0.2 # 初始化扰动
    
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置
g_vec = torch.tensor([0,0,-9.80665], device=device).unsqueeze(0).repeat(B,1)  # 重力向量向下

act_queue = [torch.zeros(B, 3, device=device) for _ in range(2)] # 动作命令队列，用于模拟控制延迟

for step in range(num_steps):

    R_camera ,T_camera = renderer.compute_view_matrix(p_ros=p, R_ros=R, camera_pitch_deg=10.0)
    rgb_images, depth_images = renderer.render(R=R_camera, T=T_camera,return_tensor=True)
    
    if step % 10 == 0:
        print(f"Step {step}: Position {p[0].detach().cpu().numpy()}")
    
    cv2.imshow('RGB', rgb_images[0].cpu().numpy()[:,:,::-1])
    cv2.imshow('Depth', (depth_images[0]/torch.max(depth_images[0])).cpu().numpy())
    if cv2.waitKey(10) & 0xFF == 27: # ESC exit
        break

    p_old = p.clone()

    dg = update_dg(dg_curr=dg, dt=dt, noise_std=0.04) # 更新扰动
    
    act_queue.append(act_pred_target) 
    current_act_cmd = act_queue.pop(0)

    p, v, a, act = simulate_position_step(
        p=p,
        v=v,
        a=a,
        R=R,
        act=act,
        act_pred=current_act_cmd, 
        dt=dt,
        enable_airmode=True,
        dg=dg,
        v_wind=torch.randn((B,3),device=device)*0.1,
        grad_decay=0.8
    )
    thrust_without_gravity = act - g_vec 
    
    R = solve_attitude_from_thrust_and_goal_vec(
        thrust_vector=thrust_without_gravity, 
        velocity=target_pos - p_old, 
        R_old=R,
        yaw_inertia=5.0, 
        dt=dt,
        yaw_ctl_delay=4.0,  # 跟着参考项目调的参数，后面可能会变
    )

    
    
cv2.destroyAllWindows()




torch.Size([4, 3])
Step 0: Position [1.5447634 1.3464289 0.3398085]
Step 10: Position [1.5467656  1.4209957  0.38248298]
Step 20: Position [1.5537866  1.8854612  0.48675442]
Step 30: Position [1.5652643  2.7962348  0.62481296]
Step 40: Position [1.5802274  4.129702   0.79350024]
Step 50: Position [1.5986782  5.855202   0.99087745]
Step 60: Position [1.6203016 7.944409  1.2145607]
Step 70: Position [ 1.6447631 10.370512   1.4633343]
Step 80: Position [ 1.6723176 13.107602   1.7352502]
Step 90: Position [ 1.7021512 16.13267    2.0281382]
Step 100: Position [ 1.7341568 19.424799   2.3397312]
Step 110: Position [ 1.7676407 22.96506    2.668526 ]
Step 120: Position [ 1.8021557 26.735      3.012515 ]
Step 130: Position [ 1.8368725 30.718353   3.3703284]
Step 140: Position [ 1.8716147 34.899307   3.7418098]
Step 150: Position [ 1.9062569 39.263855   4.126214 ]
Step 160: Position [ 1.9415475 43.799282   4.522586 ]
Step 170: Position [ 1.9780525 48.493088   4.93054  ]
Step 180: Position [ 2.015

In [3]:
# 整理





In [6]:
from drone_env import DroneSimulator

B = 4  # 批量大小
dt = 0.02  # 时间步长
num_steps = 200  # 模拟步数

# 初始化仿真环境
# mesh_path: 从 ipynb 目录看，数据在 ../data
# 完全暴露参数的版本
env = DroneSimulator(
    batch_size=B, 
    dt=dt, 
    mesh_path="../data/sample/sample.obj",
    device=device ,
    enable_airmode=True,
    enable_induced_drag=False,
    noise_std=0.04,

)

# 随机生成一个恒定的期望指令用于测试
act_pred_target = torch.rand(B, 3, device=device) * 15.0 
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置

print("Start Simulation Loop...")

for step in range(num_steps):
    # 1. 渲染 (Render)
    rgb_images, depth_images = env.render(camera_pitch=10.0)
    
    if step % 10 == 0:
        print(f"Step {step}: Position {env.p[0].detach().cpu().numpy()}")
    
    # 2. 可视化 (Visualization)
    # 注意：cv2.imshow 在某些远程/headless 环境下可能无法显示窗口
    try:
        cv2.imshow('RGB', rgb_images[0].cpu().numpy()[:,:,::-1])
        cv2.imshow('Depth', (depth_images[0]/torch.max(depth_images[0])).cpu().numpy())
        if cv2.waitKey(10) & 0xFF == 27: # ESC exit
            break
    except Exception as e:
        pass # Ignore display errors in headless

    # 3. 准备控制输入 (Control Input)
    # 原逻辑: velocity = target_pos - p_old
    # 在 step() 被调用前，env.p 即为 p_old
    target_vector = target_pos - env.p

    # 4. 模拟步进 (Simulation Step)
    state = env.step(action_cmd=act_pred_target, target_pos_vector=target_vector)

cv2.destroyAllWindows()

Loading mesh from: ../data/sample/sample.obj
Start Simulation Loop...
Step 0: Position [1.1817944  0.28814247 1.4634014 ]
Step 10: Position [1.2574636  0.35217085 1.5357882 ]
Step 20: Position [1.7313058 0.761043  1.8213274]
Step 30: Position [2.6688375 1.5722023 2.323946 ]
Step 40: Position [4.0446653 2.7635956 3.029988 ]
Step 50: Position [5.8272686 4.3088245 3.9253304]
Step 60: Position [7.9870377 6.1820655 4.9973536]
Step 70: Position [10.497636  8.359331  6.23261 ]
Step 80: Position [13.333982 10.817701  7.61949 ]
Step 90: Position [16.47203  13.536841  9.146843]
Step 100: Position [19.890184 16.49786  10.804394]
Step 110: Position [23.56721  19.68368  12.581791]
Step 120: Position [27.483576 23.0776   14.471263]
Step 130: Position [31.623182 26.664442 16.465382]
Step 140: Position [35.968678 30.429697 18.556005]
Step 150: Position [40.50547  34.360455 20.735374]
Step 160: Position [45.219307 38.44537  22.997686]
Step 170: Position [50.097137 42.673115 25.336502]
Step 180: Positio